In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

HDFS_BASE = "hdfs:///user/vagrant"

# INPUTS
UK_INPUT = f"{HDFS_BASE}/bigdata/raw/police/police_uk/london/*.csv"
NYC_INPUT = f"{HDFS_BASE}/bigdata/processed/police_data/nyc/*.csv"

# OUTPUT
POLICE_OUTPUT = f"{HDFS_BASE}/bigdata/processed/police_unified"

print("UK_INPUT :", UK_INPUT)
print("NYC_INPUT:", NYC_INPUT)
print("OUTPUT  :", POLICE_OUTPUT)

UK_INPUT : hdfs:///user/vagrant/bigdata/raw/police/police_uk/london/*.csv
NYC_INPUT: hdfs:///user/vagrant/bigdata/processed/police_data/nyc/*.csv
OUTPUT  : hdfs:///user/vagrant/bigdata/processed/police_unified


In [2]:
uk_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(UK_INPUT)
)

print("UK columns:", len(uk_raw.columns))
uk_raw.printSchema()
uk_raw.show(3, truncate=80)

UK columns: 12
root
 |-- Crime ID: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Reported by: string (nullable = true)
 |-- Falls within: string (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- LSOA code: string (nullable = true)
 |-- LSOA name: string (nullable = true)
 |-- Crime type: string (nullable = true)
 |-- Last outcome category: string (nullable = true)
 |-- Context: string (nullable = true)

+----------------------------------------------------------------+-------+---------------------------+---------------------------+---------+---------+-------------------------+---------+-------------------------+----------------------------+---------------------------+-------+
|                                                        Crime ID|  Month|                Reported by|               Falls within|Longitude| Latitude|                 Location|LSOA code|              

In [3]:
nyc_raw = spark.read.parquet(NYC_INPUT)

print("NYC columns:", len(nyc_raw.columns))
nyc_raw.printSchema()
nyc_raw.show(3, truncate=80)

NYC columns: 16
root
 |-- arrest_key: long (nullable = true)
 |-- arrest_date_id: integer (nullable = true)
 |-- pd_cd: integer (nullable = true)
 |-- pd_desc: string (nullable = true)
 |-- ky_cd: integer (nullable = true)
 |-- ky_desc: string (nullable = true)
 |-- law_cat_cd: integer (nullable = true)
 |-- arrest_boro: integer (nullable = true)
 |-- arrest_precinct: integer (nullable = true)
 |-- jurisdiction_code: integer (nullable = true)
 |-- age_group: integer (nullable = true)
 |-- perp_sex: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- city_code: string (nullable = true)
 |-- source: string (nullable = true)



[Stage 4:>                                                          (0 + 1) / 1]

+----------+--------------+-----+------------------------------+-----+---------------+----------+-----------+---------------+-----------------+---------+--------+--------+---------+---------+------+
|arrest_key|arrest_date_id|pd_cd|                       pd_desc|ky_cd|        ky_desc|law_cat_cd|arrest_boro|arrest_precinct|jurisdiction_code|age_group|perp_sex|latitude|longitude|city_code|source|
+----------+--------------+-----+------------------------------+-----+---------------+----------+-----------+---------------+-----------------+---------+--------+--------+---------+---------+------+
| 312434744|      20250905|  510|CONTROLLED SUBSTANCE, INTENT T|  117|DANGEROUS DRUGS|         1|          2|             81|                2|        3|       M|40.69306|-73.94005|      NYC|  NYPD|
| 312257501|      20250902|  511|CONTROLLED SUBSTANCE, POSSESSI|  235|DANGEROUS DRUGS|         2|          4|            110|                1|        3|       F|40.74681|-73.89175|      NYC|  NYPD|
| 313

In [4]:
# --- Geohash5 UDF (jak w Notebook 01) ---
_base32 = "0123456789bcdefghjkmnpqrstuvwxyz"

def geohash_encode(lat, lon, precision=5):
    if lat is None or lon is None:
        return None
    try:
        lat = float(lat); lon = float(lon)
    except Exception:
        return None

    lat_interval = [-90.0, 90.0]
    lon_interval = [-180.0, 180.0]
    geohash = []
    is_even = True
    bit = 0
    ch = 0
    bits = [16, 8, 4, 2, 1]

    while len(geohash) < precision:
        if is_even:
            mid = (lon_interval[0] + lon_interval[1]) / 2
            if lon > mid:
                ch |= bits[bit]
                lon_interval[0] = mid
            else:
                lon_interval[1] = mid
        else:
            mid = (lat_interval[0] + lat_interval[1]) / 2
            if lat > mid:
                ch |= bits[bit]
                lat_interval[0] = mid
            else:
                lat_interval[1] = mid

        is_even = not is_even
        if bit < 4:
            bit += 1
        else:
            geohash.append(_base32[ch])
            bit = 0
            ch = 0

    return "".join(geohash)

geohash_udf = F.udf(lambda lat, lon: geohash_encode(lat, lon, precision=5), T.StringType())


# --- helper: choose first existing column name from a list ---
def first_col(df, candidates):
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

In [5]:
uk_crime_id_col = first_col(uk_raw, ["crime_id", "Crime ID", "crimeID"])
uk_date_id_col  = first_col(uk_raw, ["date_id", "dateId", "Month", "month"])
uk_lat_col      = first_col(uk_raw, ["latitude", "Latitude"])
uk_lon_col      = first_col(uk_raw, ["longitude", "Longitude"])
uk_lsoa_code    = first_col(uk_raw, ["lsoa_code", "LSOA code", "LSOA Code"])
uk_lsoa_name    = first_col(uk_raw, ["lsoa_name", "LSOA name", "LSOA Name"])
uk_crime_type   = first_col(uk_raw, ["crime_type", "Crime type", "Crime Type"])
uk_outcome      = first_col(uk_raw, ["last_outcome_category", "Last outcome category", "Last Outcome category", "Last Outcome Category"])
uk_location     = first_col(uk_raw, ["location", "Location"])

print("UK mapping:")
print(" crime_id:", uk_crime_id_col)
print(" date_id :", uk_date_id_col)
print(" lat/lon :", uk_lat_col, uk_lon_col)
print(" lsoa    :", uk_lsoa_code, uk_lsoa_name)
print(" crime_t :", uk_crime_type)
print(" outcome :", uk_outcome)
print(" location:", uk_location)

UK mapping:
 crime_id: Crime ID
 date_id : Month
 lat/lon : Latitude Longitude
 lsoa    : LSOA code LSOA name
 crime_t : Crime type
 outcome : Last outcome category
 location: Location


In [6]:
uk = uk_raw

# latitude/longitude jako double
uk = uk.withColumn("latitude_d", F.col(uk_lat_col).cast("double")) if uk_lat_col else uk.withColumn("latitude_d", F.lit(None).cast("double"))
uk = uk.withColumn("longitude_d", F.col(uk_lon_col).cast("double")) if uk_lon_col else uk.withColumn("longitude_d", F.lit(None).cast("double"))

# date_id -> int YYYYMMDD
if uk_date_id_col is None:
    uk = uk.withColumn("date_id", F.lit(None).cast("int"))
else:
    # jeśli wygląda na YYYY-MM (string) -> YYYYMM01
    uk = uk.withColumn(
        "date_id",
        F.when(F.col(uk_date_id_col).cast("string").rlike(r"^\d{4}-\d{2}$"),
               F.regexp_replace(F.col(uk_date_id_col).cast("string"), "-", "").cast("int") * 100 + F.lit(1)  # YYYYMM01
        ).otherwise(
            F.col(uk_date_id_col).cast("int")
        )
    )

# event_id
uk = uk.withColumn("event_id", F.col(uk_crime_id_col).cast("string")) if uk_crime_id_col else uk.withColumn("event_id", F.lit(None).cast("string"))

# stałe pola identyfikujące
uk = uk.withColumn("city_code", F.lit("LON"))
uk = uk.withColumn("source", F.lit("UK_POLICE"))

# crime_type_id / crime_desc
uk = uk.withColumn("crime_type_id", F.col(uk_crime_type).cast("string")) if uk_crime_type else uk.withColumn("crime_type_id", F.lit(None).cast("string"))
uk = uk.withColumn("crime_desc", F.col(uk_location).cast("string")) if uk_location else uk.withColumn("crime_desc", F.lit(None).cast("string"))

# geohash
uk = uk.withColumn("geohash5", geohash_udf(F.col("latitude_d"), F.col("longitude_d")))

# quality filters (minimalne)
uk_clean = (
    uk.filter(F.col("event_id").isNotNull())
      .filter(F.col("latitude_d").isNotNull() & F.col("longitude_d").isNotNull())
      .filter((F.col("latitude_d") >= -90) & (F.col("latitude_d") <= 90))
      .filter((F.col("longitude_d") >= -180) & (F.col("longitude_d") <= 180))
)

print("UK rows raw  :", uk_raw.count())
print("UK rows clean:", uk_clean.count())

uk_clean.select("event_id","date_id","city_code","source","latitude_d","longitude_d","geohash5","crime_type_id","crime_desc").show(5, truncate=False)

UK rows raw  : 101615


UK rows clean: 79138


[Stage 9:>                                                          (0 + 1) / 1]

+----------------------------------------------------------------+--------+---------+---------+----------+-----------+--------+----------------------------+---------------------------+
|event_id                                                        |date_id |city_code|source   |latitude_d|longitude_d|geohash5|crime_type_id               |crime_desc                 |
+----------------------------------------------------------------+--------+---------+---------+----------+-----------+--------+----------------------------+---------------------------+
|8f0e1e2c86d619b97483658a36bd9acbbb6b34471648151aac9578e1efe1435f|20250601|LON      |UK_POLICE|50.78727  |-0.65746   |gcp8c   |Violence and sexual offences|On or near Kingsmead       |
|efc9a6a02171a5536c27cb0eac6d0bbe5ca68c5ef2a7291e2f13c9762a96bcea|20250601|LON      |UK_POLICE|50.787547 |-0.651848  |gcp8c   |Violence and sexual offences|On or near Culver Road     |
|7788cf0418f5cd470bccf17d40187c36ce22c2ed6f67f84306cce108d6b5072f|20250601|

In [7]:
nyc_arrest_key = first_col(nyc_raw, ["arrest_key", "ARREST_KEY"])
nyc_date_id    = first_col(nyc_raw, ["arrest_date_id", "ARREST_DATE_ID", "date_id"])
nyc_lat_col    = first_col(nyc_raw, ["latitude", "Latitude"])
nyc_lon_col    = first_col(nyc_raw, ["longitude", "Longitude"])
nyc_pd_desc    = first_col(nyc_raw, ["pd_desc", "PD_DESC"])
nyc_ky_desc    = first_col(nyc_raw, ["ky_desc", "KY_DESC"])
nyc_ky_cd      = first_col(nyc_raw, ["ky_cd", "KY_CD"])
nyc_pd_cd      = first_col(nyc_raw, ["pd_cd", "PD_CD"])
nyc_law_cat    = first_col(nyc_raw, ["law_cat_cd", "LAW_CAT_CD"])

print("NYC mapping:")
print(" arrest_key:", nyc_arrest_key)
print(" date_id   :", nyc_date_id)
print(" lat/lon   :", nyc_lat_col, nyc_lon_col)
print(" pd_desc   :", nyc_pd_desc)
print(" ky_desc   :", nyc_ky_desc)
print(" ky_cd/pd_cd:", nyc_ky_cd, nyc_pd_cd)
print(" law_cat_cd:", nyc_law_cat)

NYC mapping:
 arrest_key: arrest_key
 date_id   : arrest_date_id
 lat/lon   : latitude longitude
 pd_desc   : pd_desc
 ky_desc   : ky_desc
 ky_cd/pd_cd: ky_cd pd_cd
 law_cat_cd: law_cat_cd


In [8]:
nyc = nyc_raw

nyc = nyc.withColumn("latitude_d", F.col(nyc_lat_col).cast("double")) if nyc_lat_col else nyc.withColumn("latitude_d", F.lit(None).cast("double"))
nyc = nyc.withColumn("longitude_d", F.col(nyc_lon_col).cast("double")) if nyc_lon_col else nyc.withColumn("longitude_d", F.lit(None).cast("double"))

nyc = nyc.withColumn("date_id", F.col(nyc_date_id).cast("int")) if nyc_date_id else nyc.withColumn("date_id", F.lit(None).cast("int"))
nyc = nyc.withColumn("event_id", F.col(nyc_arrest_key).cast("string")) if nyc_arrest_key else nyc.withColumn("event_id", F.lit(None).cast("string"))

nyc = nyc.withColumn("city_code", F.lit("NYC"))
nyc = nyc.withColumn("source", F.lit("NYPD"))

# crime_type_id: prefer ky_cd, else pd_cd
if nyc_ky_cd:
    nyc = nyc.withColumn("crime_type_id", F.col(nyc_ky_cd).cast("string"))
elif nyc_pd_cd:
    nyc = nyc.withColumn("crime_type_id", F.col(nyc_pd_cd).cast("string"))
else:
    nyc = nyc.withColumn("crime_type_id", F.lit(None).cast("string"))

# crime_desc: prefer ky_desc, else pd_desc
if nyc_ky_desc:
    nyc = nyc.withColumn("crime_desc", F.col(nyc_ky_desc).cast("string"))
elif nyc_pd_desc:
    nyc = nyc.withColumn("crime_desc", F.col(nyc_pd_desc).cast("string"))
else:
    nyc = nyc.withColumn("crime_desc", F.lit(None).cast("string"))

nyc = nyc.withColumn("geohash5", geohash_udf(F.col("latitude_d"), F.col("longitude_d")))

nyc_clean = (
    nyc.filter(F.col("event_id").isNotNull())
       .filter(F.col("latitude_d").isNotNull() & F.col("longitude_d").isNotNull())
       .filter((F.col("latitude_d") >= -90) & (F.col("latitude_d") <= 90))
       .filter((F.col("longitude_d") >= -180) & (F.col("longitude_d") <= 180))
)

print("NYC rows raw  :", nyc_raw.count())
print("NYC rows clean:", nyc_clean.count())

nyc_clean.select("event_id","date_id","city_code","source","latitude_d","longitude_d","geohash5","crime_type_id","crime_desc").show(5, truncate=False)

NYC rows raw  : 400
NYC rows clean: 400
+---------+--------+---------+------+----------+-----------+--------+-------------+----------------------------+
|event_id |date_id |city_code|source|latitude_d|longitude_d|geohash5|crime_type_id|crime_desc                  |
+---------+--------+---------+------+----------+-----------+--------+-------------+----------------------------+
|312434744|20250905|NYC      |NYPD  |40.69306  |-73.94005  |dr5rm   |117          |DANGEROUS DRUGS             |
|312257501|20250902|NYC      |NYPD  |40.74681  |-73.89175  |dr5ry   |235          |DANGEROUS DRUGS             |
|313005693|20250916|NYC      |NYPD  |40.82867  |-73.94399  |dr72m   |106          |FELONY ASSAULT              |
|312616469|20250909|NYC      |NYPD  |40.86217  |-73.89074  |dr72q   |341          |PETIT LARCENY               |
|312491415|20250907|NYC      |NYPD  |40.75429  |-73.97294  |dr5ru   |344          |ASSAULT 3 & RELATED OFFENSES|
+---------+--------+---------+------+----------+--------

In [9]:
unified_cols = [
    "event_id",
    "date_id",
    "city_code",
    "source",
    "latitude_d",
    "longitude_d",
    "geohash5",
    "crime_type_id",
    "crime_desc",
]

uk_u  = uk_clean.select(*[c for c in unified_cols if c in uk_clean.columns])
nyc_u = nyc_clean.select(*[c for c in unified_cols if c in nyc_clean.columns])

police_unified = uk_u.unionByName(nyc_u, allowMissingColumns=True)

print("Unified rows:", police_unified.count())
police_unified.groupBy("city_code","source").count().show()
police_unified.show(5, truncate=False)

Unified rows: 79538


+---------+---------+-----+
|city_code|   source|count|
+---------+---------+-----+
|      LON|UK_POLICE|79138|
|      NYC|     NYPD|  400|
+---------+---------+-----+

+----------------------------------------------------------------+--------+---------+---------+----------+-----------+--------+----------------------------+---------------------------+
|event_id                                                        |date_id |city_code|source   |latitude_d|longitude_d|geohash5|crime_type_id               |crime_desc                 |
+----------------------------------------------------------------+--------+---------+---------+----------+-----------+--------+----------------------------+---------------------------+
|8f0e1e2c86d619b97483658a36bd9acbbb6b34471648151aac9578e1efe1435f|20250601|LON      |UK_POLICE|50.78727  |-0.65746   |gcp8c   |Violence and sexual offences|On or near Kingsmead       |
|efc9a6a02171a5536c27cb0eac6d0bbe5ca68c5ef2a7291e2f13c9762a96bcea|20250601|LON      |UK_POL

In [10]:
police_unified.select(
    F.sum(F.col("geohash5").isNull().cast("int")).alias("geohash_nulls"),
    F.sum(F.col("event_id").isNull().cast("int")).alias("event_id_nulls"),
    F.sum(F.col("date_id").isNull().cast("int")).alias("date_id_nulls"),
).show()

[Stage 28:=============================>                            (2 + 2) / 4]

+-------------+--------------+-------------+
|geohash_nulls|event_id_nulls|date_id_nulls|
+-------------+--------------+-------------+
|            0|             0|            0|
+-------------+--------------+-------------+



In [11]:
# zapis
(police_unified
 .write
 .mode("overwrite")
 .partitionBy("city_code")
 .parquet(POLICE_OUTPUT)
)

print("Saved to:", POLICE_OUTPUT)

[Stage 30:===========================================>              (3 + 1) / 4]

Saved to: hdfs:///user/vagrant/bigdata/processed/police_unified


In [12]:
police_check = spark.read.parquet(POLICE_OUTPUT)
print("Rows written:", police_check.count())
police_check.groupBy("city_code","source").count().show()
police_check.select(
    F.sum(F.col("geohash5").isNull().cast("int")).alias("geohash_nulls"),
    F.sum(F.col("event_id").isNull().cast("int")).alias("event_id_nulls"),
    F.sum(F.col("date_id").isNull().cast("int")).alias("date_id_nulls"),
).show()
police_check.show(3, truncate=False)

Rows written: 79538


+---------+---------+-----+
|city_code|   source|count|
+---------+---------+-----+
|      LON|UK_POLICE|79138|
|      NYC|     NYPD|  400|
+---------+---------+-----+

+-------------+--------------+-------------+
|geohash_nulls|event_id_nulls|date_id_nulls|
+-------------+--------------+-------------+
|            0|             0|            0|
+-------------+--------------+-------------+

+----------------------------------------------------------------+--------+---------+----------+-----------+--------+----------------------------+--------------------------+---------+
|event_id                                                        |date_id |source   |latitude_d|longitude_d|geohash5|crime_type_id               |crime_desc                |city_code|
+----------------------------------------------------------------+--------+---------+----------+-----------+--------+----------------------------+--------------------------+---------+
|8f0e1e2c86d619b97483658a36bd9acbbb6b34471648151aac95